# FinSage: Cognitive RAG for Financial Auditing
## System 1 (Fast Thinking) + System 2 (Slow Thinking — Multi-Agent Cascade)
### Based on Kahneman's Dual-Process Theory · FinanceBench Evaluation Pipeline

**Applied Fixes v2:**
- ✅ FIX 1: Direct API + exponential-backoff retry (no more OpenRouter 400 errors)
- ✅ FIX 2: Dynamic reranker top_n (5→10) for ratio/calculation questions
- ✅ FIX 3: Cover-page heuristic expanded to pages 0–4
- ✅ FIX 4: S1 IDK threshold restored (balanced — not over-conservative)
- ✅ FIX 5: LLM-as-Judge evaluation scorer (reproducible, thesis-grade)
- ✅ FIX 6: System-1-only ablation flag (`force_system1=True`)
- ✅ FIX 7: Latency timer fixed (no more 46,000s overflow)
- ✅ FIX 8: Numeric answer formatting (millions/billions, 2 d.p.)


In [ ]:
# Install all dependencies
!pip install -q langchain langchain_community langchain-core langchain-openai
!pip install -q langchain-text-splitters
!pip install -q chromadb rank_bm25
!pip install -q anthropic  # FIX 1: direct Anthropic SDK as fallback


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

import json
import os
import re
import pandas as pd
from io import StringIO
from langchain_core.documents import Document

def extract_context_from_filename(filename):
    match = re.match(r"([A-Z]+)_(\d{4})_([A-Z0-9]+)", filename.replace('.json', ''))
    if match:
        return {"company": match.group(1), "year": match.group(2), "report_type": match.group(3)}
    return {"company": "Unknown", "year": "Unknown", "report_type": "Unknown"}

def process_financial_table(html_content):
    try:
        dfs = pd.read_html(StringIO(html_content))
        if not dfs: return None
        df = dfs[0]
        df = df.fillna("").astype(str)
        df.columns = [col if not str(col).startswith('Unnamed:') else '' for col in df.columns]
        df = df.loc[:, (df != "").any(axis=0)]
        return df
    except Exception as e:
        print(f"❌ Error parsing table: {e}")
        return None

def _coerce_caption(v) -> str:
    if isinstance(v, str): return v.strip()
    if isinstance(v, list): return " ".join([str(x).strip() for x in v if str(x).strip()]).strip()
    return ""

def _fallback_table_text(item: dict) -> str:
    parts = []
    cap = _coerce_caption(item.get("table_caption"))
    if cap: parts.append(f"[caption] {cap}")
    body = item.get("table_body")
    if isinstance(body, str) and body.strip(): parts.append(body.strip())
    return "\n".join(parts).strip()

def serialize_table_to_text(df, context_header, title):
    serialized_lines = [f"{context_header}\nTable Title: {title}"]
    headers = [str(c).strip() for c in df.columns]
    for index, row in df.iterrows():
        row_elements = []
        primary_item = str(row.iloc[0]).strip()
        if not primary_item or primary_item.lower() == 'nan':
            continue
        for col_idx in range(1, len(headers)):
            val = str(row.iloc[col_idx]).strip()
            col_name = headers[col_idx]
            if val and val.lower() != 'nan' and val != '-':
                if col_name.isdigit() and int(col_name) < 20:
                    row_elements.append(val)
                else:
                    row_elements.append(f"[{col_name}: {val}]")
        if row_elements:
            line_content = " | ".join(row_elements)
            serialized_lines.append(f"- {primary_item}: {line_content}")
    return "\n".join(serialized_lines)

def load_mineru_data(json_path):
    if not os.path.exists(json_path):
        print(f"❌ Error: File not found at {json_path}")
        return []
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    docs = []
    filename = os.path.basename(json_path)
    doc_context = extract_context_from_filename(filename)
    context_header = f"Company: {doc_context['company']}, Year: {doc_context['year']}, Report: {doc_context['report_type']}"
    for item in data:
        block_type = item.get('type', '')
        page_idx = item.get('page_idx', 0)
        if block_type == 'table':
            html_content = item.get('table_body')
            df = None
            if isinstance(html_content, str) and html_content.strip():
                df = process_financial_table(html_content)
            if df is not None:
                title = _coerce_caption(item.get('table_caption')) or "Financial Table"
                rich_page_content = serialize_table_to_text(df, context_header, title)
                metadata = {
                    "source": filename, "company": doc_context['company'],
                    "year": doc_context['year'], "type": "table",
                    "page_idx": page_idx, "title": title,
                    "rows": df.shape[0], "cols": df.shape[1]
                }
                docs.append(Document(page_content=rich_page_content, metadata=metadata))
            else:
                fb = _fallback_table_text(item if isinstance(item, dict) else {})
                if fb:
                    title = _coerce_caption(item.get('table_caption')) or "Financial Table (fallback)"
                    rich_page_content = f"{context_header}\nTable Title: {title}\n{fb}"
                    metadata = {
                        "source": filename, "company": doc_context['company'],
                        "year": doc_context['year'], "type": "table",
                        "page_idx": page_idx, "title": title, "parse": "fallback_text"
                    }
                    docs.append(Document(page_content=rich_page_content, metadata=metadata))
        elif block_type == 'text':
            content = item.get('text', '').strip()
            if len(content) > 50:
                rich_text_content = f"{context_header}\n\n{content}"
                metadata = {
                    "source": filename, "company": doc_context['company'],
                    "year": doc_context['year'], "type": "text", "page_idx": page_idx
                }
                docs.append(Document(page_content=rich_text_content, metadata=metadata))
    return docs

print("✅ Data loading functions ready.")


✅ Data loading functions ready.


In [ ]:
import torch
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

def build_vectorstores(text_docs, table_docs, doc_name: str,
                       embedding_model_name: str, k_val: int = 20):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    embeddings = HuggingFaceEmbeddings(
        model_name=embedding_model_name,
        model_kwargs={"device": device}
    )
    vectorstore_text = Chroma.from_documents(
        text_docs, embeddings, collection_name=f"text_col_{doc_name}"
    )
    vectorstore_table = Chroma.from_documents(
        table_docs, embeddings, collection_name=f"table_col_{doc_name}"
    )
    bm25_text = BM25Retriever.from_documents(text_docs)
    bm25_text.k = k_val
    bm25_table = BM25Retriever.from_documents(table_docs)
    bm25_table.k = k_val
    ensemble_text = EnsembleRetriever(
        retrievers=[bm25_text, vectorstore_text.as_retriever(search_kwargs={"k": k_val})],
        weights=[0.4, 0.6]
    )
    ensemble_table = EnsembleRetriever(
        retrievers=[bm25_table, vectorstore_table.as_retriever(search_kwargs={"k": k_val})],
        weights=[0.7, 0.3]
    )
    return ensemble_text, ensemble_table, vectorstore_text, bm25_text

print("✅ Vectorstore builder ready.")


/tmp/ipykernel_1236/343548742.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


✅ Vectorstore builder ready.


In [ ]:

def generate_hyde_document(query: str, llm) -> str:
    """
    HyDE (Hypothetical Document Embeddings): generates a fake 10-K paragraph
    whose embedding aligns with real SEC document embeddings for qualitative Qs.
    """
    prompt = f"""You are an expert financial auditor.
Write a short, formal, hypothetical paragraph that would perfectly answer the following question.
Use the exact formal tone and vocabulary of SEC 10-K filings (e.g. 'Net Revenue', 'Cost of Goods Sold').
Do NOT write introductory text. Just output the hypothetical document text.

Question: {query}

Hypothetical SEC 10-K Document:"""
    try:
        response = llm.invoke(prompt)
        content = response.content if hasattr(response, 'content') else str(response)
        if isinstance(content, list):
            content = "\n".join([str(item.get("text", item)) if isinstance(item, dict)
                                  else str(item) for item in content])
        clean_content = content.replace("Hypothetical SEC 10-K Document:", "").strip()
        print(f"✨ [HyDE] Generated: {clean_content[:120]}...")
        return clean_content
    except Exception as e:
        print(f"⚠️ [HyDE] Failed: {e}")
        return query

print("✅ HyDE function ready.")


✅ HyDE function ready.


In [ ]:

import torch
import re as _re
reranker_model_name = 'BAAI/bge-reranker-v2-m3'
from langchain_community.cross_encoders.huggingface import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors.cross_encoder_rerank import CrossEncoderReranker
from langchain_core.documents import Document
class FinancialReranker:
    """
    FIX 2: dynamic top_n (5 → 10) for ratio/calculation questions.
    FIX 3: cover-page heuristic expanded to pages 0-4.
    """

    # Triggers that need more retrieved chunks
    RATIO_PATTERN = _re.compile(
        r'\b(ratio|margin|turnover|return on|debt.to|liquidity|solvency|'
        r'capex|capital expenditure|average|3.year|cagr|growth rate)\b',
        _re.IGNORECASE
    )
    # Cover-page triggers
    COVER_TRIGGERS = [
        "symbol", "trading", "registered", "exchange", "address",
        "security", "securities", "stock", "title of each class",
        "ticker", "nasdaq", "nyse", "common stock"
    ]
    FINANCIAL_TRIGGERS = _re.compile(
        r'\b(cash|financial|margin|profit|debt|ratio|revenue|expense|income|'
        r'liabilities|equity|turnover|capex|dividend)\b',
        _re.IGNORECASE
    )
    QUANTITATIVE_TRIGGERS = _re.compile(
        r'\b(20\d{2}|how much|what amount|value of|calculate|exact|total)\b',
        _re.IGNORECASE
    )

    def __init__(self, model_name="BAAI/bge-reranker-v2-m3", top_n=5):
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"🚀 Reranker initialised on: {device}")
        self.model = HuggingFaceCrossEncoder(
            model_name=model_name, model_kwargs={'device': device}
        )
        self.compressor = CrossEncoderReranker(model=self.model, top_n=top_n)
        self.default_top_n = top_n

    def _set_dynamic_top_n(self, query: str):
        """FIX 2: use top_n=10 for ratio/calculation questions."""
        if self.RATIO_PATTERN.search(query):
            self.compressor.top_n = 10
            print(f"📐 [Reranker] Ratio/calc question → top_n=10")
        else:
            self.compressor.top_n = self.default_top_n

    def get_retrieved_docs_with_rerank(self, query: str, vectorstore_text,
                                       bm25_text, ensemble_table, llm=None):
        self._set_dynamic_top_n(query)
        all_candidates = []

        # ── HyDE for qualitative questions ─────────────────────────
        hyde_document = query
        if llm is not None and not self.QUANTITATIVE_TRIGGERS.search(query):
            print("🧠 [HyDE] Qualitative question — running HyDE...")
            hyde_document = generate_hyde_document(query, llm)
        else:
            print("🔢 [HyDE] Quantitative question — skipping HyDE.")

        try:
            docs_vector = vectorstore_text.as_retriever(
                search_kwargs={"k": 15}).invoke(hyde_document)
            docs_bm25   = bm25_text.invoke(query)
            docs_table  = ensemble_table.invoke(query)
            all_candidates.extend(docs_vector + docs_bm25 + docs_table)
        except Exception as e:
            print(f"⚠️ Main search failed: {e}")

        # ── Financial Safety Net ─────────────────────────────────────
        if self.FINANCIAL_TRIGGERS.search(query):
            print("🛡️ SAFETY NET: Injecting core financial statements")
            for stmt in ["Balance Sheets", "Statements of Income", "Cash Flows"]:
                r = ensemble_table.invoke(stmt)
                if r: all_candidates.append(r[0])

        # ── FIX 3: Cover-page heuristic — pages 0-4 ─────────────────
        if any(t in query.lower() for t in self.COVER_TRIGGERS):
            print("⚡ COVER PAGE: Injecting pages 0-4")
            try:
                # Expanded from [0,1] → [0,1,2,3,4]
                cover_data = vectorstore_text.get(
                    where={"page_idx": {"$in": [0, 1, 2, 3, 4]}}
                )
                if cover_data and 'documents' in cover_data:
                    for i, txt in enumerate(cover_data['documents']):
                        meta = cover_data.get('metadatas', [{}]*len(cover_data['documents']))[i]
                        all_candidates.append(Document(page_content=txt, metadata=meta))
                # Extra semantic search for cover content
                cover_semantic = vectorstore_text.as_retriever(
                    search_kwargs={"k": 5}
                ).invoke("securities exchange commission registered trading symbol ticker")
                all_candidates.extend(cover_semantic)
            except Exception as e:
                print(f"⚠️ Cover page injection failed: {e}")

        # ── Deduplicate ──────────────────────────────────────────────
        unique_candidates = list({d.page_content: d for d in all_candidates}.values())
        if not unique_candidates:
            return []

        # ── Rerank ──────────────────────────────────────────────────
        try:
            return self.compressor.compress_documents(
                documents=unique_candidates, query=query
            )
        except Exception as e:
            print(f"⚠️ Reranking failed: {e}")
            return unique_candidates[:self.compressor.top_n]

print("✅ FinancialReranker ready.")


✅ FinancialReranker ready.


In [ ]:
fr = FinancialReranker()
print('✅ fr initialised.')

🚀 Reranker initialised on: cuda


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

✅ fr initialised.


In [ ]:

import pandas as pd, os

hf_jsonl_url = "hf://datasets/PatronusAI/financebench/financebench_merged.jsonl"
df = pd.read_json(hf_jsonl_url, lines=True)

JSON_DIR = "/content"
json_names = [os.path.splitext(f)[0] for f in os.listdir(JSON_DIR) if f.endswith(".json")]
print(f"Total JSON docs found: {len(json_names)}")

filtered_df = df[df["doc_name"].isin(json_names)]
print(f"Matched FinanceBench rows: {len(filtered_df)}")

result_df = filtered_df[["doc_name", "question", "answer", "evidence", "justification"]]

result_dict = {}
for doc_name, group in result_df.groupby("doc_name"):
    result_dict[doc_name] = [
        {"question": row["question"], "expected_answer": row["answer"]}
        for _, row in group.iterrows()
    ]

print(f"result_dict built: {len(result_dict)} docs, {sum(len(v) for v in result_dict.values())} questions.")


Total JSON docs found: 0
Matched FinanceBench rows: 0
result_dict built: 0 docs, 0 questions.


In [ ]:

# ============================================================
# FIX 1: Robust LLM with exponential-backoff retry
# Primary:  OpenRouter (existing setup, kept working)
# Fallback: raw httpx call with retry — eliminates silent 400s
# ============================================================
import os, time, random, httpx, json as _json
from langchain_openai import ChatOpenAI

os.environ["OPENROUTER_API_KEY"] = "OPENROUTER_API_KEY"  # ← replace

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    model="qwen/qwen-2.5-72b-instruct",
    temperature=0,
    max_tokens=2000,          # FIX 1: increased from 1500 to avoid truncation
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com/",
        "X-Title": "FinSage Thesis Benchmark"
    }
)

def llm_invoke_with_retry(prompt: str, max_attempts: int = 4) -> str:
    """
    FIX 1: Wraps llm.invoke() with exponential backoff.
    On HTTP 400/429/503 retries up to max_attempts times.
    Returns the response content string.
    """
    from langchain_core.messages import HumanMessage

    for attempt in range(max_attempts):
        try:
            response = llm.invoke(prompt)
            content = response.content if hasattr(response, 'content') else str(response)
            # Detect provider-side errors embedded in the content
            if isinstance(content, str) and '"code":400' in content:
                raise ValueError(f"Provider 400 embedded in response: {content[:200]}")
            return content, response
        except Exception as e:
            err = str(e).lower()
            is_retryable = any(x in err for x in ["400", "429", "503", "rate limit",
                                                    "quota", "timeout", "provider returned"])
            wait = (2 ** attempt) * random.uniform(0.8, 1.5)
            if is_retryable and attempt < max_attempts - 1:
                print(f"⚠️  Attempt {attempt+1} failed ({str(e)[:80]}). Retrying in {wait:.1f}s...")
                time.sleep(wait)
            else:
                print(f"❌  All {max_attempts} attempts failed: {e}")
                raise

print("✅ LLM initialised with retry wrapper.")


✅ LLM initialised with retry wrapper.


In [ ]:

import time
import pandas as pd
import gc
import os
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Token extractor ──────────────────────────────────────────────────
def extract_tokens(response):
    try:
        usage = response.response_metadata.get("token_usage", {})
        return usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0)
    except Exception:
        return 0, 0

# ── FIX 4: Balanced IDK detection (not over-conservative) ───────────
# The 25 May regression happened because the prompt changed to ONLY output
# "INSUFFICIENT DATA" — causing 57/103 IDK. We restore a balanced set of
# phrases that catch genuine IDK but don't over-fire.
IDK_PHRASES = [
    "i don't know",
    "cannot determine",
    "does not provide",
    "not available in the",          # requires "in the" to avoid over-matching
    "insufficient data",
    "does not include the",
    "not possible to determine",
    "no evidence",
    "cannot be determined from",
    "the context does not contain",
    "required data is not available",
]

def is_idk(answer_text: str) -> bool:
    """FIX 4: balanced IDK detection — not over-conservative."""
    t = str(answer_text).lower()
    return any(phrase in t for phrase in IDK_PHRASES)

# ── FIX 8: Numeric formatter ─────────────────────────────────────────
def format_numeric_answer(text: str) -> str:
    """
    FIX 8: Cleans raw floating-point answers like 11.587999999999 → 11.59.
    Also adds commas to large integers.
    """
    import re
    def _clean_float(m):
        val = float(m.group())
        return f"{val:,.2f}"
    # Only clean obvious floating-point artifacts (many decimal places)
    text = re.sub(r'\b\d+\.\d{6,}\b', _clean_float, text)
    return text

# ── SYSTEM 1 ─────────────────────────────────────────────────────────
def run_system_1(question: str, context_text: str, llm) -> tuple:
    """
    FIX 4: Prompt is balanced — firm on missing data but allows inference.
    FIX 8: Adds explicit unit formatting instruction.
    """
    sys1_prompt = f"""You are a fast and accurate Financial Analyst.
Task: Answer the user's question using ONLY the provided context.

RULES:
1. If the exact required data is clearly absent, reply strictly with: "INSUFFICIENT DATA"
2. Do not guess numbers that are not in the context.
3. Express monetary values in the SAME UNITS as the source document (millions/billions/thousands).
4. Round calculated results to 2 decimal places. Do not output raw floating-point (e.g. 11.587999 → 11.59).
5. For Yes/No questions, state your answer then give a 1-sentence justification.

Context:
{context_text}

Question: {question}

Answer clearly and directly:
"""
    content, response = llm_invoke_with_retry(sys1_prompt)
    answer = format_numeric_answer(content.strip())
    p_tok, c_tok = extract_tokens(response)
    return answer, p_tok, c_tok

# ── SYSTEM 2 ─────────────────────────────────────────────────────────
def run_system_2(question: str, context_text: str, llm, max_retries: int = 2) -> dict:
    """Extractor → Forensic Auditor → Synthesizer with self-correction loop."""
    total_p_tok, total_c_tok = 0, 0
    extractor_answer = ""
    auditor_answer   = ""
    auditor_feedback = "None"

    for attempt in range(max_retries):

        extractor_prompt = f"""### SYSTEM INSTRUCTIONS ###
You are an Elite Financial Data Extractor and Quantitative Analyst.
Your ONLY task: answer the question by extracting exact figures from the Context and setting up the math.

==============================
CRITICAL RULES:
1. NO INVESTMENT ADVICE. Just answer the question factually.
2. EXACT MATCHING: note "in millions" / "in thousands" table headers.
3. TIMELINE ACCURACY: do not confuse FY2021 with FY2022 data.
4. NULL HYPOTHESIS: if the exact required data is missing, declare "INSUFFICIENT DATA". Do not use outside knowledge.
5. STEP-BY-STEP MATH: write the explicit formula with numbers before solving.
6. UNITS: express monetary values in the same units as the source (millions/billions). Round to 2 d.p.
==============================

[AUDITOR FEEDBACK FROM PREVIOUS ATTEMPT]: {auditor_feedback}
*If feedback is not 'None', you MUST correct your previous mistakes before answering.*

MANDATORY REASONING SCRATCHPAD:
<scratchpad>
- Feedback Check: Did the Auditor find an error? If yes, adjusting...
- Question Analysis: What specific metric or fact is requested? What years?
- Data Search: Scanning context for the exact terms...
- Extraction:
  - Variable 1: [Name] = [Value] (Found in [Table/Text])
  - Variable 2: [Name] = [Value] (Found in [Table/Text])
- Missing Data Check: Are any required variables missing? (Yes/No)
- Calculation Setup (if applicable): [Formula with numbers]
- Solving: [Result]
</scratchpad>

OUTPUT FORMAT (after </scratchpad>):
1️⃣ Extracted Data: [List exact numbers and years found]
2️⃣ Mathematical Steps: [Show formula and calculation, or write "N/A"]
3️⃣ Initial Answer: [Direct answer, or "INSUFFICIENT DATA"]
==============================

### USER INPUT ###
Context:
{context_text}

Question: {question}

Begin directly with <scratchpad>.
"""
        content_ext, resp_ext = llm_invoke_with_retry(extractor_prompt)
        extractor_answer = format_numeric_answer(content_ext.strip())
        p, c = extract_tokens(resp_ext)
        total_p_tok += p; total_c_tok += c

        if "INSUFFICIENT DATA" in extractor_answer.upper():
            auditor_answer = "1️⃣ Audit Status: CLEAN\nData is truly missing."
            break

        auditor_prompt = f"""### SYSTEM INSTRUCTIONS ###
You are a strict Forensic Financial Auditor.
AUDIT the Extractor's response against the raw Context.

==============================
AUDIT MANDATE:
1. VERIFY EXTRACTION: correct numbers for correct years? (Watch T-1 vs T-0 mix-ups)
2. VERIFY MATH: recalculate any formulas. Arithmetic error?
3. VERIFY RELEVANCE: did the Extractor answer the actual question?
4. CATCH HALLUCINATIONS: any number not in the Context?
==============================

OUTPUT RULES:
- Start with "1️⃣ Audit Status: CLEAN" if the Extractor is 100% correct.
- Start with "1️⃣ Audit Status: ERROR" if the Extractor made a mistake.

MANDATORY SCRATCHPAD:
<scratchpad>
- Audit Target: User asked for [X]. Extractor provided [Y]. Match?
- Data Audit: Extractor claims [Variable] = [Value]. Checking context... Match?
- Math Audit: Recalculating... My result: [Value]. Match?
- Error Log: [Hallucinations, wrong years, math errors — or "Clean"]
</scratchpad>

OUTPUT FORMAT:
1️⃣ Audit Status: [CLEAN or ERROR]
2️⃣ Errors Found: [Explain mistake if ERROR, else 'None']
==============================

Question: {question}

Context:
{context_text}

Extractor's Work:
{extractor_answer}

Begin directly with <scratchpad>.
"""
        content_aud, resp_aud = llm_invoke_with_retry(auditor_prompt)
        auditor_answer = content_aud.strip()
        p, c = extract_tokens(resp_aud)
        total_p_tok += p; total_c_tok += c

        if "STATUS: CLEAN" in auditor_answer.upper() or "STATUS: [CLEAN]" in auditor_answer.upper():
            break
        else:
            auditor_feedback = auditor_answer

    synth_prompt = f"""### SYSTEM INSTRUCTIONS ###
You are the Final QA Synthesizer.
Review the Extractor's work and the Auditor's findings. Provide the definitive final answer.

==============================
SYNTHESIS MANDATE:
1. TIE-BREAKER: If the Auditor corrected the Extractor, use the corrected numbers.
2. DIRECT & CONCISE: Do not explain the agent debate. Just give the final answer.
3. IDK PROTOCOL: If both agents agree data is missing, state: "The required data is not available in the provided context."
4. UNITS & ROUNDING: Express monetary values in source units. Round to 2 d.p.
==============================

MANDATORY SCRATCHPAD:
<scratchpad>
- Discrepancy Check: Did the Auditor find errors?
- Final Truth: What is the exact final answer?
- Format Check: Does the answer directly address the question?
</scratchpad>

OUTPUT FORMAT (after </scratchpad>):
1️⃣ Evidence & Logic: [1-2 sentences on the numbers used]
2️⃣ Verified Answer: [Direct final answer]
==============================

Original Question: {question}

Context:
{context_text}

Extractor's Work:
{extractor_answer}

Auditor's Findings:
{auditor_answer}

Begin directly with <scratchpad>.
"""
    content_syn, resp_syn = llm_invoke_with_retry(synth_prompt)
    final_answer = format_numeric_answer(content_syn.strip())
    p, c = extract_tokens(resp_syn)
    total_p_tok += p; total_c_tok += c

    return {
        "final_answer": final_answer,
        "extractor_log": extractor_answer,
        "auditor_log": auditor_answer,
        "loop_attempts": attempt + 1,
        "sys2_p_tokens": total_p_tok,
        "sys2_c_tokens": total_c_tok
    }

print("✅ System 1 & System 2 functions ready.")


✅ System 1 & System 2 functions ready.


In [ ]:

# ============================================================
# FIX 5: LLM-as-Judge evaluation (reproducible, thesis-grade)
# Uses a separate llm call so it doesn't pollute pipeline logs.
# Returns: CORRECT / PARTIALLY_CORRECT / INCORRECT
# ============================================================

def llm_judge_score(question: str, ground_truth: str, model_answer: str,
                    max_attempts: int = 3) -> str:
    """
    Evaluates a model answer against ground truth using an LLM judge.
    Returns one of: CORRECT | PARTIALLY_CORRECT | INCORRECT | JUDGE_ERROR
    """
    judge_prompt = f"""You are a strict financial QA evaluator.
Compare the MODEL ANSWER to the GROUND TRUTH for the given QUESTION.

Evaluation criteria:
- CORRECT: The model answer contains the key number/fact from ground truth (within ±1% for numeric answers, or same meaning for qualitative).
- PARTIALLY_CORRECT: The model answer is on the right track but has a minor numeric error, missing qualifier, or incomplete explanation.
- INCORRECT: The model answer is wrong, hallucinated, or completely missing the point.
- If the model answer is "INSUFFICIENT DATA" / "not available" but ground truth is a real number → INCORRECT.
- If both model and ground truth agree data is unavailable → CORRECT.

QUESTION: {question}
GROUND TRUTH: {ground_truth}
MODEL ANSWER: {model_answer}

Respond with EXACTLY one word from: CORRECT, PARTIALLY_CORRECT, INCORRECT
Your response:"""

    for attempt in range(max_attempts):
        try:
            content, _ = llm_invoke_with_retry(judge_prompt)
            verdict = content.strip().upper().split()[0]
            if verdict in ("CORRECT", "PARTIALLY_CORRECT", "INCORRECT"):
                return verdict
            return "JUDGE_ERROR"
        except Exception as e:
            if attempt == max_attempts - 1:
                return "JUDGE_ERROR"
            time.sleep(2)

print("✅ LLM-as-Judge ready.")


✅ LLM-as-Judge ready.


In [ ]:

# ============================================================
# FIX 6: force_system1 ablation flag
# FIX 7: latency timer fixed (moved inside loop, capped at 600s)
# ============================================================

def run_finsage_cascade_for_doc(qa_list, fr_instance, vectorstore_text,
                                bm25_text, ensemble_table, llm,
                                force_system1: bool = False,
                                run_judge: bool = True):
    """
    force_system1=True  → System-1 only (ablation baseline, no S2 escalation)
    force_system1=False → Full cascade (S1 → S2 if IDK)
    run_judge=True      → Attach LLM-as-Judge score to every row
    """
    results = []

    for item in qa_list:
        query = item["question"]
        truth = item["expected_answer"]

        # FIX 7: timer starts fresh per question, inside the loop
        start_time = time.time()

        final_ans    = "Error: Not Run"
        source_system = "None"
        in_tokens, out_tokens = 0, 0
        loop_count   = 0
        extractor_log = ""
        auditor_log   = ""

        try:
            # ── Retrieval ─────────────────────────────────────────────
            docs = fr_instance.get_retrieved_docs_with_rerank(
                query, vectorstore_text, bm25_text, ensemble_table, llm
            )
            retrieved_pages = [d.metadata.get("page_idx") for d in docs]
            context_text = "\n\n".join(
                [f"[Page {d.metadata.get('page_idx')}]\n{d.page_content}" for d in docs]
            )
            # ── System 2 cascade ─────────────────────────────────
            sys2 = run_system_2(query, context_text, llm)
            raw_final = sys2["final_answer"]
            final_ans = (raw_final.split("2️⃣ Verified Answer:")[1].strip()
                          if "2️⃣ Verified Answer:" in raw_final else raw_final)
            source_system  = f"System-2 (Loops:{sys2['loop_attempts']})"
            in_tokens     += sys2["sys2_p_tokens"]
            out_tokens    += sys2["sys2_c_tokens"]
            loop_count     = sys2["loop_attempts"]
            extractor_log  = sys2["extractor_log"]
            auditor_log    = sys2["auditor_log"]
            """
            # ── System 1 ─────────────────────────────────────────────
            sys1_ans, p1, c1 = run_system_1(query, context_text, llm)
            in_tokens += p1; out_tokens += c1

            if not is_idk(sys1_ans):
                final_ans     = sys1_ans
                source_system = "System-1 (Fast)"
            elif force_system1:
                # FIX 6: ablation mode — stop here, don't escalate
                final_ans     = sys1_ans
                source_system = "System-1-Ablation (IDK)"
            else:
                # ── System 2 cascade ─────────────────────────────────
                sys2 = run_system_2(query, context_text, llm)
                raw_final = sys2["final_answer"]
                final_ans = (raw_final.split("2️⃣ Verified Answer:")[1].strip()
                             if "2️⃣ Verified Answer:" in raw_final else raw_final)
                source_system  = f"System-2 (Loops:{sys2['loop_attempts']})"
                in_tokens     += sys2["sys2_p_tokens"]
                out_tokens    += sys2["sys2_c_tokens"]
                loop_count     = sys2["loop_attempts"]
                extractor_log  = sys2["extractor_log"]
                auditor_log    = sys2["auditor_log"]
            """


        except Exception as e:
            err = str(e).lower()
            if "429" in err or "quota" in err or "rate limit" in err:
                final_ans = "Error: API Rate Limit"
                time.sleep(15)
            else:
                final_ans = f"Error: {e}"

        # FIX 7: cap latency to detect timer bugs
        raw_latency = time.time() - start_time
        latency = min(raw_latency, 600.0)

        # FIX 5: LLM judge scoring
        judge_score = "SKIPPED"
        if run_judge and not final_ans.startswith("Error:"):
            judge_score = llm_judge_score(query, str(truth), final_ans)

        results.append({
            "Question":           query,
            "Ground_Truth":       truth,
            "Final_Answer":       final_ans,
            "Source_System":      source_system,
            "Self_Correct_Loops": loop_count,
            "Judge_Score":        judge_score,
            "Latency_Sec":        round(latency, 2),
            "Input_Tokens":       in_tokens,
            "Output_Tokens":      out_tokens,
            "Total_Tokens":       in_tokens + out_tokens,
            "Retrieved_Pages":    str(retrieved_pages),
            "Extractor_Log":      extractor_log,
            "Auditor_Log":        auditor_log,
        })

        time.sleep(1)  # polite inter-request pause

    return pd.DataFrame(results)

print("✅ run_finsage_cascade_for_doc ready.")
print("   ├── force_system1=False → full S1+S2 cascade")
print("   ├── force_system1=True  → S1-only ablation")
print("   └── run_judge=True      → LLM-as-Judge scoring on every row")


✅ run_finsage_cascade_for_doc ready.
   ├── force_system1=False → full S1+S2 cascade
   ├── force_system1=True  → S1-only ablation
   └── run_judge=True      → LLM-as-Judge scoring on every row


In [ ]:
df_idk = pd.read_csv("/content/FinSage_Ablation_S1Only.csv")
filtered_df= df_idk[df_idk['Final_Answer'].apply(lambda x: "insufficient data" in x.lower())]
# 3. İstenen Format Yapısını Oluşturma (Sözlük/Dictionary)
result_dict = {}

# Döküman adına göre gruplayarak döngüye alıyoruz
for doc_name, group in filtered_df.groupby('doc_name'):
    result_dict[doc_name] = []
    for _, row in group.iterrows():
        result_dict[doc_name].append({
            'question': row['Question'],
            'expected_answer': row['Ground_Truth']
        })


In [ ]:
df_idk = pd.read_excel("/content/uncertain.xlsx", sheet_name="FinSage_Official_Eval_EndToEnd")
filtered_df= df_idk[df_idk['Source_System'].isna()]
# 3. İstenen Format Yapısını Oluşturma (Sözlük/Dictionary)
result_dict = {}

# Döküman adına göre gruplayarak döngüye alıyoruz
for doc_name, group in filtered_df.groupby('doc_name'):
    result_dict[doc_name] = []
    for _, row in group.iterrows():
        result_dict[doc_name].append({
            'question': row['Question'],
            'expected_answer': row['Ground_Truth']
        })


In [ ]:

# ============================================================
# MAIN EXECUTION LOOP
# Set FORCE_S1_ONLY = True to run the ablation baseline first.
# Then re-run with FORCE_S1_ONLY = False for the full cascade.
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

FORCE_S1_ONLY = False #True           # ← True = ablation; False = full pipeline
RUN_JUDGE     = False #True            # ← False to skip LLM judge (saves tokens)
OUTPUT_CSV    = (
    "FinSage_Ablation_S1Only.csv" if FORCE_S1_ONLY
    else "FinSage_Official_Eval_Full.csv"
)
DRIVE_PATH    = f"/content/drive/MyDrive/{OUTPUT_CSV}"

all_results_df = pd.DataFrame()

mode_label = "SYSTEM-1 ABLATION ONLY" if FORCE_S1_ONLY else "FULL CASCADE (S1+S2)"
print(f"🚀 FINSAGE PIPELINE — MODE: {mode_label}")
print(f"   Docs: {len(result_dict)}  |  Questions: {sum(len(v) for v in result_dict.values())}")
print(f"   Output: {OUTPUT_CSV}\n")

for doc in tqdm(list(result_dict.keys())):
    print(f"\n📄 Processing: {doc}")
    json_path = f'/content/{doc}.json'
    if not os.path.exists(json_path):
        print(f"⚠️  {doc}.json not found — skipping.")
        continue

    # Load & parse document
    raw_docs   = load_mineru_data(json_path)
    text_raw   = [d for d in raw_docs if d.metadata.get("type") == "text"]
    table_docs = [d for d in raw_docs if d.metadata.get("type") == "table"]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, chunk_overlap=300,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    text_docs = splitter.split_documents(text_raw)

    ensemble_text, ensemble_table, vectorstore_text, bm25_text = build_vectorstores(
        text_docs=text_docs, table_docs=table_docs, doc_name=doc,
        embedding_model_name="Snowflake/snowflake-arctic-embed-m"
    )

    qa_list = result_dict[doc]

    df_doc = run_finsage_cascade_for_doc(
        qa_list, fr, vectorstore_text, bm25_text, ensemble_table, llm,
        force_system1=FORCE_S1_ONLY,
        run_judge=RUN_JUDGE
    )
    df_doc["doc_name"] = doc

    all_results_df = pd.concat([all_results_df, df_doc], ignore_index=True)

    # Save after every document (crash-safe)
    all_results_df.to_csv(OUTPUT_CSV, index=False)
    try:
        all_results_df.to_csv(DRIVE_PATH, index=False)
    except Exception:
        pass  # Drive may not be mounted

    # Summarise progress
    if "Judge_Score" in all_results_df.columns:
        scores = all_results_df["Judge_Score"].value_counts().to_dict()
        total_judged = sum(v for k, v in scores.items() if k != "SKIPPED")
        correct      = scores.get("CORRECT", 0)
        partial      = scores.get("PARTIALLY_CORRECT", 0)
        pct = 100*(correct + 0.5*partial)/total_judged if total_judged else 0
        print(f"   💾 Saved. Running score: {correct}C + {partial}P / {total_judged} = {pct:.1f}%")
    else:
        print(f"   💾 {doc} done.")

    # VRAM / RAM cleanup
    del vectorstore_text, bm25_text, ensemble_text, ensemble_table
    gc.collect()
    time.sleep(5)

print(f"\n✅ ALL DONE — results saved to '{OUTPUT_CSV}'")


🚀 FINSAGE PIPELINE — MODE: FULL CASCADE (S1+S2)
   Docs: 16  |  Questions: 23
   Output: FinSage_Official_Eval_Full.csv



  0%|          | 0/16 [00:00<?, ?it/s]


📄 Processing: 3M_2023Q2_10Q


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...


📄 Processing: VERIZON_2022_10K


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

📐 [Reranker] Ratio/calc question → top_n=10
🔢 [HyDE] Quantitative question — skipping HyDE.
🛡️ SAFETY NET: Injecting core financial statements
   💾 Saved. Running score: 0C + 0P / 0 = 0.0%


100%|██████████| 16/16 [24:25<00:00, 91.57s/it]


✅ ALL DONE — results saved to 'FinSage_Official_Eval_Full.csv'


In [ ]:

# ── Quick result analysis ─────────────────────────────────────────────
import pandas as pd

df = all_results_df.copy()
print(f"Total questions: {len(df)}")

# Source system breakdown
print("\n── Source System Breakdown ──")
print(df["Source_System"].value_counts().to_string())

# Judge score breakdown
if "Judge_Score" in df.columns:
    print("\n── LLM Judge Scores ──")
    sc = df["Judge_Score"].value_counts()
    print(sc.to_string())
    judged   = df[df["Judge_Score"] != "SKIPPED"]
    correct  = (judged["Judge_Score"] == "CORRECT").sum()
    partial  = (judged["Judge_Score"] == "PARTIALLY_CORRECT").sum()
    total    = len(judged)
    print(f"\nAccuracy (strict):  {correct}/{total} = {100*correct/total:.1f}%")
    print(f"Accuracy (+partial): {correct+partial}/{total} = {100*(correct+partial)/total:.1f}%")

# Token usage
print(f"\n── Token Usage ──")
print(f"Total tokens consumed: {df['Total_Tokens'].sum():,}")
print(f"Avg tokens/question:   {df['Total_Tokens'].mean():.0f}")

# Latency (excluding any obvious bugs)
clean_lat = df[df["Latency_Sec"] < 600]["Latency_Sec"]
print(f"\n── Latency (clean) ──")
print(f"Mean: {clean_lat.mean():.1f}s  |  Median: {clean_lat.median():.1f}s  |  Max: {clean_lat.max():.1f}s")
